# Writing exercise: Section 5 answers (local, not for students)

Worked versions of the optional tasks in `writing.ipynb`. On a student clone, run `writing.ipynb` through Section 3 first. On the `answers` branch, the second
code cell builds Andrew's finished code and makes `form-tutorial-writing.root` itself.
Use the phaindia kernel and open this notebook from `writing/`.

Every task writes its own jsonnet copy and output file, so `form-tutorial-writing.root` is untouched.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

import numpy as np

ROOT_LIBDIR = subprocess.check_output(["root-config", "--libdir"], text=True).strip()
if ROOT_LIBDIR not in sys.path:
    sys.path.insert(0, ROOT_LIBDIR)
import ROOT
from IPython.display import Image, display

WRITING_DIR = Path.cwd()
assert (WRITING_DIR / "write_wires.jsonnet").exists(), "Open this notebook from the writing/ directory"
PREFIX = WRITING_DIR.parent / "opt"


def prepend(var, path):
    entries = [e for e in os.environ.get(var, "").split(":") if e]
    if str(path) not in entries:
        os.environ[var] = ":".join([str(path)] + entries)


for var, sub in [("LD_LIBRARY_PATH", "lib"), ("PATH", "bin"), ("PHLEX_PLUGIN_PATH", "lib")]:
    prepend(var, PREFIX / sub)
ROOT.gSystem.Load(str(PREFIX / "lib" / "libwire_readout.so"))

WIRES = "root_rntuple__wires_provider:simulated_wires__sim"
FIELD = "sim._0.waveform_"
N_WIRES = 1024


def read_wires(file_name, entries):
    """Waveforms for the given entry numbers (entry = channel + 1024 * APA for spill 0)."""
    reader = ROOT.RNTupleReader.Open(WIRES, file_name)
    view = reader.GetView["std::vector<uint16_t>"](FIELD)
    out = [np.array(view(e), dtype=float) for e in entries]
    del view, reader
    return out


def draw_wires(waveforms, labels, png, title="Waveforms"):
    canvas = ROOT.TCanvas(png, "", 1000, 450)
    legend = ROOT.TLegend(0.70, 0.75, 0.89, 0.89)
    colours = [ROOT.kBlue + 1, ROOT.kOrange + 7, ROOT.kGreen + 2, ROOT.kMagenta + 1]
    hists = []
    top = max(w.max() for w in waveforms) * 1.1
    for i, (w, label) in enumerate(zip(waveforms, labels)):
        h = ROOT.TH1D(f"{png}_{i}", f"{title};Time tick;ADC", len(w), 0, len(w))
        for tick, value in enumerate(w):
            h.SetBinContent(tick + 1, value)
        h.SetLineColor(colours[i % len(colours)])
        h.SetMaximum(top)
        h.Draw("hist" if i == 0 else "hist same")
        legend.AddEntry(h, label, "l")
        hists.append(h)
    legend.Draw()
    canvas.Print(png)
    display(Image(png))


def run_variant(name, edits):
    """Copy write_wires.jsonnet with regex edits, run it, return the output file name."""
    text = Path("write_wires.jsonnet").read_text()
    for pattern, repl in edits:
        text, n = re.subn(pattern, repl, text, count=1)
        assert n == 1, f"no match for {pattern}"
    jsonnet = f"write_wires_{name}.jsonnet"
    Path(jsonnet).write_text(text)
    log = f"write_wires_{name}.log"
    result = subprocess.run(["phlex", "-c", jsonnet, "-j", "1", "--stage", "sim"],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT)
    print(f"{jsonnet}: phlex exit code {result.returncode} (log in {log})")
    return re.search(r"output_file:\s*'([^']+)'", text).group(1)


ADD_TO_PROVIDER = r"(wires_provider:\s*{\s*cpp:\s*'simulate_wires_provider',)"
print("ready")

In [ ]:
# Build the tutorial and make form-tutorial-writing.root if they are missing.
# On the answers branch (Andrew's finished main + notebooks) this makes the notebook self-contained.
BUILD_DIR = PREFIX / "build"
if not (PREFIX / "lib" / "libwire_readout.so").exists():
    BUILD_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["cmake", str(WRITING_DIR.parent), f"-DCMAKE_INSTALL_PREFIX={PREFIX}", "-DCMAKE_BUILD_TYPE=Release"],
                   cwd=BUILD_DIR, check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["make", "-j4", "install"], cwd=BUILD_DIR, check=True, stdout=subprocess.DEVNULL)
    ROOT.gSystem.Load(str(PREFIX / "lib" / "libwire_readout.so"))
if not Path("form-tutorial-writing.root").exists():
    subprocess.run(["phlex", "-c", "write_wires.jsonnet", "-j", "1", "--stage", "sim"],
                   check=True, stdout=open("write_wires.log", "w"), stderr=subprocess.STDOUT)
print("built:", (PREFIX / "lib" / "libwire_readout.so").exists(), "| input file:", Path("form-tutorial-writing.root").exists())

## Task 1: look around the file

Finds the wire in APA 13 with the tallest pulse and the one with the smallest maximum, and draws them
next to the same channel in APA 0.

In [ ]:
apa13 = read_wires("form-tutorial-writing.root", [ch + 13 * N_WIRES for ch in range(N_WIRES)])
peaks = np.array([w.max() for w in apa13])
loud, quiet = int(peaks.argmax()), int(peaks.argmin())
print(f"APA 13: tallest pulse on channel {loud} ({peaks[loud]:.0f} ADC), quietest wire is channel {quiet} ({peaks[quiet]:.0f} ADC)")
print(f"{(peaks > 100).sum()} of {N_WIRES} wires go above 100 ADC, the top of the noise")

apa0_loud = read_wires("form-tutorial-writing.root", [loud])[0]
draw_wires([apa13[loud], apa13[quiet], apa0_loud],
           [f"APA 13 ch {loud}", f"APA 13 ch {quiet}", f"APA 0 ch {loud}"], "task1_wires.png")

## Task 2a: a fixed seed

Two runs with `seed: 42` give identical waveforms. Without a seed, each run differs.

In [ ]:
seed_edits = [(ADD_TO_PROVIDER, r"\1\n      seed: 42,"),
              (r"form-tutorial-writing\.root", "form-tutorial-seed.root")]
first = read_wires(run_variant("seed", seed_edits), [243 + 13 * N_WIRES])[0]
second = read_wires(run_variant("seed", seed_edits), [243 + 13 * N_WIRES])[0]
print("seed 42, two runs identical:", np.array_equal(first, second))

noseed_edits = [(r"form-tutorial-writing\.root", "form-tutorial-noseed.root")]
a = read_wires(run_variant("noseed", noseed_edits), [243 + 13 * N_WIRES])[0]
b = read_wires(run_variant("noseed", noseed_edits), [243 + 13 * N_WIRES])[0]
print("no seed, two runs identical:", np.array_equal(a, b))
draw_wires([first, a, b], ["seed 42", "no seed, run 1", "no seed, run 2"], "task2a_seed.png", "APA 13, channel 243")

## Task 2b: `background_height: 10`

Same seed as 2a, so the only change is the height. The noise doubles, from 0 to 100 ADC to 0 to 200,
and the signal heights double too, because `simulate_wires.cpp` draws them from 50 to 100 times
`background_height × n_noise_functions`.

In [ ]:
tall_edits = [(ADD_TO_PROVIDER, r"\1\n      seed: 42,\n      background_height: 10,"),
              (r"form-tutorial-writing\.root", "form-tutorial-bh10.root")]
tall = read_wires(run_variant("bh10", tall_edits), [243 + 13 * N_WIRES])[0]
print(f"max ADC: height 5 -> {first.max():.0f}, height 10 -> {tall.max():.0f} (ratio {tall.max() / first.max():.2f})")
draw_wires([first, tall], ["background_height 5", "background_height 10"], "task2b_height.png", "APA 13, channel 243")

## Task 3: TTree or RNTuple

In [ ]:
ttree = run_variant("ttree", [(r"'ROOT_RNTUPLE'", "'ROOT_TTREE'"),
                              (r"form-tutorial-writing\.root", "form-tutorial-ttree.root")])
for name in ["form-tutorial-writing.root", ttree]:
    print(f"{name:32} {Path(name).stat().st_size / 1e6:7.1f} MB")

## Task 4: every ADC value in one APA

A bell shape between 0 and 100 ADC from the ten sine waves, and a long tail from the signal pulses.
The dashed line is the reading exercise's pedestal, 60 ADC: everything above it in the noise becomes
a "hit".

In [ ]:
values = np.concatenate(apa13)
hist = ROOT.TH1D("all_adc", "Every ADC value, APA 13;ADC;Ticks", 300, 0, 300)
hist.FillN(len(values), values, ROOT.nullptr)

canvas = ROOT.TCanvas("c4", "", 900, 500)
canvas.SetLogy()
hist.Draw("hist")
line = ROOT.TLine(60, 1, 60, hist.GetMaximum())
line.SetLineStyle(2)
line.SetLineColor(ROOT.kRed)
line.Draw()
canvas.Print("task4_all_adc.png")
print(f"mean {values.mean():.1f} ADC, {100 * (values > 60).mean():.1f}% of ticks above 60, {100 * (values > 100).mean():.2f}% above 100")
Image("task4_all_adc.png")